# Classificação de transações: exploração

Lê o CSV rotulado (`ml/dados/transacoes_rotuladas.csv`: as colunas do parser + `categoria`) e separa X e y.

> Antes de fazer commit deste notebook, limpe as saídas (**Clear All Outputs**): elas podem conter dados derivados dos extratos.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_validate, train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# funciona com o notebook aberto em ml/ ou na raiz do projeto
CSV = next(p for p in (Path("dados/transacoes_rotuladas.csv"), Path("ml/dados/transacoes_rotuladas.csv")) if p.exists())

# keep_default_na=False: contraparte vazia continua "" (sem contraparte), não vira NaN
df = pd.read_csv(CSV, keep_default_na=False, parse_dates=["data"])
df["valor"] = df["valor"].astype(float)
print(df.shape)
df.dtypes

In [ ]:
ALVO = "categoria"

y = df[ALVO]
X = df.drop(columns=[ALVO])

print("X:", X.shape, "| y:", y.shape)
y.value_counts()

In [ ]:
print(X.columns)

In [ ]:
# --- variáveis categóricas e numéricas -> matriz só de números para os modelos ---

# id_usuario não entra no modelo (usuários futuros não estão no treino):
# fica guardado para a validação cruzada por usuário (GroupKFold)
grupos = X["id_usuario"]

CATEGORICAS = ["historico", "meio", "tipo", "banco", "contraparte"]
NUMERICAS = ["valor_log", "dia_semana", "dia_mes", "mes"]

feat = X[CATEGORICAS].copy()
feat["valor_log"] = np.log1p(X["valor"].abs())  # o sinal já está em "tipo"; o log reduz a cauda longa
feat["dia_semana"] = X["data"].dt.day_of_week     # 0 = segunda ... 6 = domingo
feat["dia_mes"] = X["data"].dt.day
feat["mes"] = X["data"].dt.month

preprocessador = ColumnTransformer(
    [
        # one-hot: uma coluna 0/1 por valor; o que aparece menos de 5 vezes vira uma coluna "infrequent"
        # (importante para contraparte, com ~1.200 valores, e para lojas novas na aplicação)
        ("cat", OneHotEncoder(min_frequency=2, handle_unknown="infrequent_if_exist", sparse_output=False),
         CATEGORICAS),
        # min-max: numéricas na faixa [0, 1]
        ("num", MinMaxScaler(), NUMERICAS),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X = preprocessador.fit_transform(feat)

print("X transformado:", X.shape)
for nome, _, cols in preprocessador.transformers_:
    if nome == "cat":
        enc = preprocessador.named_transformers_["cat"]
        for c, cats in zip(cols, enc.categories_):
            print(f"  {c}: {len(cats)} valores -> {sum(1 for f in X.columns if f.startswith(c + '_'))} colunas")
    elif nome == "num":
        print(f"  numéricas: {len(cols)} colunas")

In [ ]:
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_validate, train_test_split


X_train, X_test, Y_train, Y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

model.fit(X_train,Y_train)
y_pred = model.predict(X_test)

print(accuracy_score(Y_test,y_pred))


In [ ]:
from sklearn.base import clone


def cross_validation(model, X, y, n_splits=5, groups=None):
    """Acurácia e F1 macro médios (± desvio padrão) em validação cruzada.

    groups=None: StratifiedKFold embaralhado (mantém a proporção das classes em cada fold).
    groups=...:  GroupKFold (o mesmo grupo, ex. usuário, nunca fica no treino e no teste ao mesmo tempo).
    """
    if groups is None:
        kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        divisoes = kf.split(X, y)
    else:
        divisoes = GroupKFold(n_splits=n_splits).split(X, y, groups)

    acc, f1 = [], []
    for train_index, test_index in divisoes:
        m = clone(model)  # modelo "zerado" a cada fold
        m.fit(X.iloc[train_index], y.iloc[train_index])
        y_pred = m.predict(X.iloc[test_index])
        acc.append(accuracy_score(y.iloc[test_index], y_pred))
        f1.append(f1_score(y.iloc[test_index], y_pred, average="macro"))

    print(f"Acurácia = {np.mean(acc):.3f} ± {np.std(acc):.3f}")
    print(f"F1 macro = {np.mean(f1):.3f} ± {np.std(f1):.3f}")
    return acc, f1

In [ ]:
rfc = RandomForestClassifier(random_state=42)

cross_validation(rfc,X,y)

In [ ]:
# mesma validação, mas por usuário: o modelo é testado em usuários que não viu no treino (caso real)
cross_validation(rfc, X, y, groups=grupos)

In [ ]:
# --- comparação de modelos: validação aleatória x validação por usuário ---
import io
import time
from contextlib import redirect_stdout

from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier

modelos = {
    "Regressão logística": LogisticRegression(max_iter=2000),
    "SVM linear": LinearSVC(),
    "Naive Bayes (Complement)": ComplementNB(),  # aceita porque o X é todo >= 0 (one-hot + min-max)
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Árvore de decisão": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
}

resultados = []
for nome, modelo in modelos.items():
    inicio = time.perf_counter()
    with redirect_stdout(io.StringIO()):  # a função imprime; aqui só guardamos os números
        acc_alea, f1_alea = cross_validation(modelo, X, y)
        acc_grupo, f1_grupo = cross_validation(modelo, X, y, groups=grupos)
    resultados.append({
        "modelo": nome,
        "acurácia (aleatório)": np.mean(acc_alea),
        "F1 macro (aleatório)": np.mean(f1_alea),
        "acurácia (por usuário)": np.mean(acc_grupo),
        "F1 macro (por usuário)": np.mean(f1_grupo),
        "± F1 (por usuário)": np.std(f1_grupo),
        "tempo (s)": time.perf_counter() - inicio,
    })
    print(f"{nome}: ok")

# ordenado pelo F1 macro por usuário: a estimativa mais próxima da aplicação real
comparacao = pd.DataFrame(resultados).set_index("modelo").sort_values("F1 macro (por usuário)", ascending=False)
comparacao.round(3)

In [ ]:
# --- SVM linear (TF-IDF na contraparte, sem data): matriz de confusão em heatmap ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict
from sklearn.svm import LinearSVC

svm = Pipeline([
    ("prep", ColumnTransformer([
        ("contraparte", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True),
         "contraparte"),
        ("cat", OneHotEncoder(min_frequency=5, handle_unknown="infrequent_if_exist"),
         ["historico", "meio", "tipo", "banco"]),
        ("num", MinMaxScaler(), ["valor_log"]),
    ])),
    ("clf", LinearSVC()),
])

# cada linha recebe a previsão de um modelo que NÃO a viu no treino (5 folds)
validacoes = {
    "aleatória (StratifiedKFold)": dict(cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)),
    "por usuário (GroupKFold)": dict(cv=GroupKFold(n_splits=5), groups=grupos),
}
classes_ordem = sorted(y.unique())

fig, eixos = plt.subplots(1, 2, figsize=(18, 7.5))
for eixo, (nome, args) in zip(eixos, validacoes.items()):
    y_pred = cross_val_predict(svm, feat, y, **args)
    acc, f1 = accuracy_score(y, y_pred), f1_score(y, y_pred, average="macro")
    # normalize="true": cada linha soma 100% -> mostra quanto de cada classe REAL foi para cada classe prevista
    ConfusionMatrixDisplay.from_predictions(y, y_pred, labels=classes_ordem, normalize="true",
                                            values_format=".0%", cmap="Blues", colorbar=False,
                                            xticks_rotation=45, ax=eixo)
    eixo.set_title(f"Validação {nome}\nacurácia {acc:.1%} | F1 macro {f1:.1%}")
    eixo.set_xlabel("previsto")
    eixo.set_ylabel("real")
    print(f"\n=== {nome} ===")
    print(classification_report(y, y_pred, digits=3))

plt.tight_layout()
plt.show()